# RQ V: Order sensitivity


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown

repo = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "reported_runs/analysis").is_dir())
sys.path.insert(0, str(repo)) if str(repo) not in sys.path else None
from reported_runs.analysis.presentation import load_results, NotebookReport, temporal_table, effect_table, rank_summary, unique_trajectories, label_conditions
from reported_runs.analysis.plotting import trajectory_grid, effect_grid, rank_grid, association_grid
from reported_runs.analysis.comparisons import summarise_iterations
%matplotlib inline

results = load_results()
report = NotebookReport(results, 'rq5')
scales = report.scales
trajectory = label_conditions(results["trajectory"])
sources = results["sources"]
summaries = results["summaries"]
effect_summaries = results["effect_summaries"]
pd.set_option("display.max_columns", 18)
pd.set_option("display.max_rows", 60)

## Predictor and outcome


In [ ]:
from reported_runs.analysis.sections_rq4_5 import order_plot_wording
order_pairs = results['order_pairs']
order_correlations = results['order_correlations']
report.table(order_pairs, 'rq5_order_sensitivity_pairs', "")

## Exploratory association


In [ ]:
available_pairs = order_pairs[order_pairs['available']]
for family in available_pairs['predictor_family'].unique():
    selected = available_pairs[available_pairs['predictor_family'].eq(family) & available_pairs['window'].eq('mean_1_30')]
    wording = order_plot_wording(family, outcome_window='mean_1_30')
    fig = association_grid(selected, title=wording['title'], x='baseline_order_sensitivity', y='head_click_reduction_pp', x_label=wording['x_label'], y_label=wording['y_label'])
    name = 'computational' if family.startswith('Computational') else 'llm'
    report.figure(fig, selected, f'rq5_{name}_sensitivity_association', "", recommended=False, why="")

In [ ]:
report.table(order_correlations, 'rq5_within_seed_associations', "")

## Replica agreement and shuffling eligibility


In [ ]:
control_ids = sources[sources['study'].eq('movielens_mitigation_a300_e804') & sources['intervention'].eq('control')][['study', 'condition_id', 'seed']]
baseline_probes = results['probes'].merge(control_ids, on=['study', 'condition_id', 'seed'], validate='many_to_one')
probe_summary = summarise_iterations(baseline_probes, [m for m in ['replica_agreement', 'shuffled_agreement', 'agreement_sensitivity', 'replica_pair_agreement', 'shuffle_tvd', 'triple_valid_clicks', 'tvd_opportunities'] if m in baseline_probes])
report.table(temporal_table(probe_summary, probe_summary['metric'].unique().tolist()), 'rq5_predictor_eligibility', "")

## Endpoint companion


In [ ]:
endpoint_pairs = available_pairs[available_pairs['window'].eq('iteration_30')]
for family in endpoint_pairs['predictor_family'].unique():
    selected = endpoint_pairs[endpoint_pairs['predictor_family'].eq(family)]
    wording = order_plot_wording(family, outcome_window='iteration_30')
    fig = association_grid(selected, title=wording['title'], x='baseline_order_sensitivity', y='head_click_reduction_pp', x_label=wording['x_label'], y_label=wording['y_label'])
    name = 'computational' if family.startswith('Computational') else 'llm'
    report.figure(fig, selected, f'rq5_{name}_endpoint_association', "")